## Step 1: Parse the PDF into structured elements

In [23]:
from unstructured.partition.pdf import partition_pdf

elements = partition_pdf(
    filename="sample_report.pdf",
    strategy="hi_res",              # layout-detection model, not just text extraction — this is what lets it recognize Table/Image/Title as distinct categories
    infer_table_structure=True,     # required for el.metadata.text_as_html to be populated on Table elements
    extract_images_in_pdf=True,
    extract_image_block_output_dir="./extracted_images"  # detected images are saved here; el.metadata.image_path points to the saved file
)

for i, el in enumerate(elements):
    print(f"[{i}] {el.category} | page {el.metadata.page_number} | {str(el.text)[:60] if el.text else '[image]'}")

No languages specified, defaulting to English.


[0] Title | page 1 | Q3 2026 Financial Summary
[1] NarrativeText | page 1 | This report summarizes quarterly performance across all oper
[2] NarrativeText | page 1 | Revenue growth this quarter was driven primarily by three fa
[3] NarrativeText | page 1 | Regionally, the Americas remained the largest contributor to
[4] NarrativeText | page 1 | On the cost side, total operating expenses grew more slowly 
[5] NarrativeText | page 1 | Looking ahead to Q4, management expects revenue growth to re
[6] Title | page 2 | Regional Revenue Breakdown
[7] NarrativeText | page 2 | Revenue grew across all regions this quarter, though the pac
[8] NarrativeText | page 2 | APAC revenue grew eight percent quarter over quarter to four
[9] NarrativeText | page 2 | EMEA revenue grew five percent quarter over quarter to three
[10] NarrativeText | page 2 | Americas revenue grew six percent quarter over quarter to fi
[11] NarrativeText | page 2 | Taken together, the regional results reflect a business that
[12

In [24]:
# look up by category, not position — element indices shift whenever the PDF content changes
table_el = next(el for el in elements if el.category == "Table")
print(table_el.metadata.text_as_html)

<table><thead><tr><th></th><th>Revenue ($M)</th><th>Growth (%)</th></tr></thead><tbody><tr><td>APAC</td><td>4.2</td><td>8%</td></tr><tr><td>EMEA</td><td>3.1</td><td>5%</td></tr><tr><td>Americas</td><td>5.6</td><td>6%</td></tr></tbody></table>


## Step 2: Enrich non-text elements (table summary + image caption)

In [25]:
def summarize_table(html):
    # Replace with real LLM call: llm.generate(f"Summarize this table: {html}")
    return "Table showing regional revenue and growth: APAC 4.2M (8%), EMEA 3.1M (5%), Americas 5.6M (6%)."

def caption_image(image_path):
    # Replace with real vision LLM call
    return "Bar chart titled Figure 1 showing Q3 revenue by region: APAC $4.2M, EMEA $3.1M, Americas $5.6M."

for el in elements:
    if el.category == "Table":
        el.metadata.generated_summary = summarize_table(el.metadata.text_as_html)   # new attribute, not a built-in unstructured field — read back in Step 3
    if el.category == "Image":
        el.metadata.generated_caption = caption_image(el.metadata.image_path)       # same idea — read back in Step 3

In [26]:
# same category lookup as Step 1's table-html check — avoids hardcoding indices that shift if the PDF changes
table_el = next(el for el in elements if el.category == "Table")
image_el = next(el for el in elements if el.category == "Image")
print(table_el.metadata.generated_summary)
print(image_el.metadata.generated_caption)

Table showing regional revenue and growth: APAC 4.2M (8%), EMEA 3.1M (5%), Americas 5.6M (6%).
Bar chart titled Figure 1 showing Q3 revenue by region: APAC $4.2M, EMEA $3.1M, Americas $5.6M.


## Step 3 : Convert enriched elements → LlamaIndex Documents, grouped by page/section

In [27]:
from llama_index.core import Document
from collections import defaultdict

page_text = defaultdict(list)

for el in elements:
    page = el.metadata.page_number
    if el.category == "Table":
        page_text[page].append(f"[Table]: {el.metadata.generated_summary}")     # swap the raw table for its text summary — keeps the embedded text purely textual
    elif el.category == "Image":
        page_text[page].append(f"[Figure]: {el.metadata.generated_caption}")    # same idea for images
    elif el.text:
        page_text[page].append(el.text)

documents = [
    Document(text="\n\n".join(texts), metadata={"page": page})   # one Document per page — this is the unit HierarchicalNodeParser splits next
    for page, texts in sorted(page_text.items())
]

print(documents[1].text)  # page 2 preview

Regional Revenue Breakdown

Revenue grew across all regions this quarter, though the pace and drivers of growth varied considerably by market. The table below shows the breakdown by region, and Figure 1 visualizes the same data as a bar chart. Americas posted the highest absolute revenue, while APAC saw the fastest growth rate of any region.

APAC revenue grew eight percent quarter over quarter to four point two million dollars, the fastest growth rate of any region. This growth was driven primarily by strong mobile subscriber additions in Southeast Asia and continued expansion of cloud service adoption among mid-market customers in Japan and South Korea. Favorable currency movements against the US dollar added a modest additional tailwind. Management expects APAC growth to remain elevated next quarter as the recently expanded local sales team ramps up productivity.

EMEA revenue grew five percent quarter over quarter to three point one million dollars. Growth was supported by several 

## Step 4: Apply HierarchicalNodeParser

In [28]:
from llama_index.core.node_parser import HierarchicalNodeParser, get_leaf_nodes

# chunk_sizes ordered biggest -> smallest: 1024-token "parent" chunks, each further split into ~256-token "leaf" chunks
node_parser = HierarchicalNodeParser.from_defaults(chunk_sizes=[1024, 256])
hierarchical_nodes = node_parser.get_nodes_from_documents(documents)   # every parent AND every leaf, all levels, linked by relationship metadata
leaf_nodes = get_leaf_nodes(hierarchical_nodes)                        # just the smallest chunks — only these get embedded in Step 5

print(f"Total nodes (parent+leaf): {len(hierarchical_nodes)}, leaf nodes: {len(leaf_nodes)}")

Total nodes (parent+leaf): 11, leaf nodes: 7


## Step 5: Store

In [29]:
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.core.storage.docstore import SimpleDocumentStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings

chroma_client = chromadb.PersistentClient(path="./chroma_db")   # writes to disk — survives a kernel restart
chroma_collection = chroma_client.get_or_create_collection("pdf_report")

vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

# Doc store still separate — holds full node content + parent/child relationships.
# Unlike the Chroma collection above, this is in-memory only and is NOT persisted to disk here,
# so parent lookups in Step 6 only work within the current kernel session.
docstore = SimpleDocumentStore()
docstore.add_documents(hierarchical_nodes)   # ALL nodes (parents + leaves) — Chroma below only ever sees leaves

storage_context = StorageContext.from_defaults(
    vector_store=vector_store,
    docstore=docstore
)

Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")   # 384-dim embeddings, downloaded once then cached locally
index = VectorStoreIndex(leaf_nodes, storage_context=storage_context)   # only leaf_nodes get embedded — parents are looked up later, never embedded
## To get previously stored indexs
# index = VectorStoreIndex.from_vector_store(vector_store)

print("Stored", chroma_collection.count(), "vectors in ChromaDB")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7969.24it/s]


Stored 7 vectors in ChromaDB


## Step 6: Retrieve with auto-merging

In [30]:
from llama_index.core.retrievers import AutoMergingRetriever

# Both queries target the SAME parent — the "Regional Revenue Breakdown" page (page 2), which was
# split into exactly 2 leaf children: leaf A (intro + APAC + EMEA) and leaf B (Americas + table).
# That makes the merge math binary here: 1 of 2 matched -> ratio 0.5 -> NOT merged (0.5 is not > 0.5).
#                                         2 of 2 matched -> ratio 1.0 -> MERGED.

narrow_query = "What was APAC's revenue growth this quarter?"
# -> only matches leaf A -> 1/2 -> stays as a small leaf chunk

broad_query = "How did revenue perform across all regions this quarter?"
# -> matches BOTH leaf A and leaf B -> 2/2 -> collapses into the single parent chunk

base_retriever = index.as_retriever(similarity_top_k=2)
# verbose=True prints "> Merging N nodes into parent node." exactly when a merge happens below
merging_retriever = AutoMergingRetriever(base_retriever, storage_context, verbose=True)

for label, query in [("NARROW", narrow_query), ("BROAD", broad_query)]:
    print(f"\n=== {label}: {query!r} ===")

    leaf_results = base_retriever.retrieve(query)      # always leaf-sized, never merges
    merged_results = merging_retriever.retrieve(query)  # may replace leaves with their parent

    print(f"base retriever    -> {len(leaf_results)} node(s), sizes={[len(n.node.text) for n in leaf_results]} chars")
    print(f"merging retriever -> {len(merged_results)} node(s), sizes={[len(n.node.text) for n in merged_results]} chars")

    # fewer nodes coming back than went in is the tell — siblings got collapsed into 1 parent node
    merged = len(merged_results) < len(leaf_results)
    print("RESULT:", "MERGED into parent chunk" if merged else "stayed as leaf(ves) — no merge")


=== NARROW: "What was APAC's revenue growth this quarter?" ===
base retriever    -> 2 node(s), sizes=[1371, 1392] chars
merging retriever -> 2 node(s), sizes=[1371, 1392] chars
RESULT: stayed as leaf(ves) — no merge

=== BROAD: 'How did revenue perform across all regions this quarter?' ===
> Merging 2 nodes into parent node.
> Parent node id: c93bbbf6-e5fc-4a81-82a6-28888c39d667.
> Parent node text: Regional Revenue Breakdown

Revenue grew across all regions this quarter, though the pace and dri...

base retriever    -> 2 node(s), sizes=[1371, 891] chars
merging retriever -> 1 node(s), sizes=[2264] chars
RESULT: MERGED into parent chunk
